# Sesión 4 — Introducción a pandas
## Series, DataFrames y la anatomía de un análisis real

**Contexto clínico:** Perfiles bioquímicos y hematológicos caninos  
**Herramientas:** Python 3.10 · pandas · NumPy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print(f"pandas:  {pd.__version__}")
print(f"NumPy:   {np.__version__}")

## 1. Series — array de NumPy con etiquetas

Una Series es la unidad básica de pandas.  
Internamente es un array de NumPy, pero con un **índice** que etiqueta cada valor.  
Piénsala como una columna de Excel que recuerda el nombre de cada fila.

In [ ]:
# Crear una Series desde una lista
# pandas asigna automáticamente un índice numérico (0, 1, 2...)

hemoglobina = pd.Series([15.1, 11.8, 17.2, 9.5, 16.0, 12.5, 14.2, 10.2, 16.5, 13.8])

print(hemoglobina)
print(f"\nTipo:   {type(hemoglobina)}")
print(f"dtype:  {hemoglobina.dtype}")
print(f"Shape:  {hemoglobina.shape}")

In [ ]:
# El índice es lo que diferencia una Series de un array NumPy
# Puedes definir etiquetas significativas en lugar de números

hemoglobina_etiquetada = pd.Series(
    data  = [15.1, 11.8, 17.2, 9.5, 16.0],
    index = ['C001', 'C002', 'C003', 'C004', 'C005'],
    name  = 'hemoglobina_g_dL'    # nombre del analito — aparece al imprimir
)

print(hemoglobina_etiquetada)
print(f"\nNombre:  {hemoglobina_etiquetada.name}")
print(f"Índice:  {hemoglobina_etiquetada.index.tolist()}")
print(f"Valores: {hemoglobina_etiquetada.values}")   # array NumPy puro

In [ ]:
# Acceso a elementos de una Series

# Por etiqueta de índice
print(f"Hgb de C002: {hemoglobina_etiquetada['C002']} g/dL")

# Por posición numérica
print(f"Tercer valor: {hemoglobina_etiquetada.iloc[2]} g/dL")

# Filtro booleano — exactamente igual que NumPy
anemia = hemoglobina_etiquetada < 12.0
print(f"\nPacientes con anemia (Hgb < 12 g/dL):")
print(hemoglobina_etiquetada[anemia])

In [ ]:
# Las operaciones sobre Series son vectorizadas — igual que NumPy
# Pero la Series conserva el índice en el resultado

hemoglobina_mmol = hemoglobina_etiquetada * 0.621   # g/dL → mmol/L
print("Hemoglobina en mmol/L:")
print(hemoglobina_mmol.round(2))
# Observa: el índice C001–C005 se conserva automáticamente

# Estadísticas sobre Series
print(f"\nMedia:   {hemoglobina_etiquetada.mean():.2f} g/dL")
print(f"Mediana: {hemoglobina_etiquetada.median():.2f} g/dL")
print(f"SD:      {hemoglobina_etiquetada.std():.2f} g/dL")
print(f"Min:     {hemoglobina_etiquetada.min():.2f} g/dL")
print(f"Max:     {hemoglobina_etiquetada.max():.2f} g/dL")

## 2. DataFrame — la tabla central del análisis

Un DataFrame es una colección de Series que comparten el mismo índice.  
Piénsalo como una tabla de base de datos: filas = registros, columnas = variables.  
Cada columna es una Series independiente. El índice es el identificador de cada fila.

In [ ]:
# Forma 1: desde un diccionario — la más común al construir datos manualmente
# Cada clave es el nombre de columna, cada valor es la lista de datos

datos = {
    'paciente_id':         ['C001', 'C002', 'C003', 'C004', 'C005',
                            'C006', 'C007', 'C008', 'C009', 'C010'],
    'raza':                ['Labrador', 'Beagle', 'Pastor Alemán', 'Bulldog', 'Golden',
                            'Schnauzer', 'Poodle', 'Boxer', 'Dálmata', 'Husky'],
    'edad_años':           [3, 7, 5, 4, 2, 9, 6, 8, 3, 5],
    'peso_kg':             [28.5, 12.0, 35.2, 22.1, 30.0, 8.5, 6.2, 29.8, 24.5, 27.3],
    'sexo':                ['M', 'H', 'M', 'M', 'H', 'H', 'M', 'H', 'M', 'H'],

    # Serie roja
    'eritrocitos_mill_uL': [6.2, 4.8, 7.1, 3.9, 6.5, 5.2, 5.8, 4.1, 6.8, 5.5],
    'hemoglobina_g_dL':    [15.1, 11.8, 17.2, 9.5, 16.0, 12.5, 14.2, 10.2, 16.5, 13.8],
    'hematocrito_pct':     [45.2, 35.4, 51.8, 28.6, 48.0, 37.5, 42.6, 30.8, 49.5, 41.4],
    'VCM_fL':              [72.9, 73.8, 72.9, 73.3, 73.8, 72.1, 73.4, 75.1, 72.8, 75.3],

    # Serie blanca
    'leucocitos_mil_uL':   [8.5, 12.3, 7.2, 18.6, 9.1, 11.4, 8.8, 15.2, 7.9, 10.3],
    'neutrofilos_pct':     [68, 75, 65, 82, 70, 78, 67, 80, 66, 72],
    'linfocitos_pct':      [22, 16, 25, 10, 20, 14, 24, 12, 25, 18],

    # Plaquetas
    'plaquetas_mil_uL':    [245, 189, 310, 98, 278, 220, 265, 142, 290, 235],

    # Bioquímica básica
    'glucosa_mg_dL':       [95, 102, 88, 145, 91, 110, 87, 132, 93, 98],
    'bun_mg_dL':           [18, 22, 15, 45, 19, 28, 16, 38, 17, 21],
    'creatinina_mg_dL':    [0.9, 1.1, 0.8, 2.8, 1.0, 1.3, 0.9, 2.1, 0.8, 1.0],
    'alt_U_L':             [32, 45, 28, 89, 35, 67, 30, 112, 29, 41],

    'diagnostico':         ['Normal', 'Anemia leve', 'Normal',
                            'Anemia moderada + AZO', 'Normal', 'Normal',
                            'Normal', 'Anemia leve + AZO', 'Normal', 'Normal']
}

df = pd.DataFrame(datos)
print(f"DataFrame creado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

In [ ]:
# Forma 2: desde arrays NumPy
# Útil cuando ya tienes los datos como arrays

eritrocitos = np.array([6.2, 4.8, 7.1, 3.9, 6.5])
hemoglobina = np.array([15.1, 11.8, 17.2, 9.5, 16.0])
hematocrito = np.array([45.2, 35.4, 51.8, 28.6, 48.0])
ids         = ['C001', 'C002', 'C003', 'C004', 'C005']

df_numpy = pd.DataFrame({
    'paciente_id':         ids,
    'eritrocitos_mill_uL': eritrocitos,
    'hemoglobina_g_dL':    hemoglobina,
    'hematocrito_pct':     hematocrito
})

print("DataFrame desde NumPy:")
print(df_numpy)

In [ ]:
# Forma 3: desde CSV — la más común en trabajo real
# Primero guardamos el DataFrame principal a CSV, luego lo volvemos a cargar
# Esto simula el flujo real: recibes un archivo, lo cargas, lo analizas

import os
os.makedirs('../datos', exist_ok=True)

# Guardar
df.to_csv('../datos/perfil_clinico_canino.csv', index=False)

# Cargar — pd.read_csv() es la función que más vas a usar en tu carrera
df_cargado = pd.read_csv('../datos/perfil_clinico_canino.csv')

print(f"Cargado desde CSV: {df_cargado.shape}")
print(f"Son idénticos: {df.equals(df_cargado)}")

## 3. El índice — el identificador de cada fila

El índice es lo que hace que pandas sea diferente a una matriz NumPy.  
Es la "dirección" de cada fila: permite acceder, alinear y combinar datos por etiqueta.  
Por defecto es 0, 1, 2... pero puede ser cualquier cosa: IDs, fechas, nombres.

Este concepto no existe en Java. Es nuevo y vale la pena entenderlo bien.

In [ ]:
# El índice por defecto es numérico (RangeIndex)
print("Índice por defecto:")
print(df.index)

# Cambiar el índice al ID del paciente — muy común en datos clínicos
# Así cada fila tiene una etiqueta significativa
df_indexado = df.set_index('paciente_id')

print("\nCon paciente_id como índice:")
print(df_indexado.index)
print(f"\nShape: {df_indexado.shape}")   # ahora tiene 1 columna menos
df_indexado.head(3)

In [ ]:
# Con el índice como paciente_id, el acceso es mucho más claro

# Obtener todos los datos de C004
datos_c004 = df_indexado.loc['C004']
print("Resultados completos de C004:")
print(datos_c004)

In [ ]:
# reset_index() devuelve el índice a columna normal
# Útil cuando necesitas el índice como dato accesible
df_reset = df_indexado.reset_index()
print("Después de reset_index():")
print(df_reset[['paciente_id', 'hemoglobina_g_dL', 'diagnostico']].head())

# Nota importante: set_index() NO modifica df en su lugar a menos que uses inplace=True
# Las operaciones de pandas devuelven un nuevo DataFrame por defecto
# Esta es la diferencia más importante con Java donde los métodos suelen modificar el objeto

print(f"\ndf original sigue intacto: 'paciente_id' en columnas = {'paciente_id' in df.columns}")

## 4. Protocolo de inspección — lo primero que haces con cualquier dataset

En laboratorio, antes de analizar una muestra verificas su identidad, volumen y aspecto.  
En datos haces lo mismo: verificar estructura, tipos, completitud y valores extremos.

In [ ]:
# Trabajamos con df (con índice numérico por defecto, paciente_id como columna)
print("=" * 50)
print("PROTOCOLO DE INSPECCIÓN DE DATASET")
print("=" * 50)

# 1. Dimensiones
print(f"\n1. DIMENSIONES")
print(f"   Filas (pacientes):   {df.shape[0]}")
print(f"   Columnas (variables): {df.shape[1]}")

In [ ]:
# 2. Primeras y últimas filas
print("2. PRIMERAS 3 FILAS")
df.head(3)

In [ ]:
print("3. ÚLTIMAS 3 FILAS")
df.tail(3)

In [ ]:
# 4. Tipos de datos y valores nulos — la inspección más importante
print("4. TIPOS DE DATOS Y VALORES NO NULOS")
df.info()

In [ ]:
# 5. Estadísticas descriptivas automáticas
# Solo columnas numéricas por defecto
print("5. ESTADÍSTICAS DESCRIPTIVAS")
df.describe().round(2)

In [ ]:
# 6. Verificación de valores nulos por columna
print("6. VALORES NULOS POR COLUMNA")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos ✓")

In [ ]:
# 7. Valores únicos en columnas categóricas
print("7. VARIABLES CATEGÓRICAS")
categoricas = ['sexo', 'diagnostico']

for col in categoricas:
    print(f"\n  {col}:")
    print(f"  {df[col].value_counts().to_dict()}")

In [ ]:
# 8. Muestra aleatoria — ver filas al azar, no siempre las primeras
print("8. MUESTRA ALEATORIA (3 filas)")
df.sample(3, random_state=42)

## 5. `.loc` e `.iloc` — selección precisa de datos

pandas tiene dos sistemas de indexación:

- **`.loc`** — selección por **etiqueta** (label): nombre de columna, valor del índice
- **`.iloc`** — selección por **posición** (integer location): número de fila/columna

Cuando el índice es numérico (0, 1, 2...), `.loc` y `.iloc` parecen iguales pero no lo son.  
La diferencia se vuelve crítica cuando el índice tiene etiquetas o cuando haces slicing.

In [ ]:
# .iloc — siempre por posición numérica, como un array
# [fila, columna] — ambos son enteros

print("=== .iloc — por posición ===")

# Fila 0 (primera fila), todas las columnas
print("\nFila 0 completa:")
print(df.iloc[0])

In [ ]:
# Filas 0 a 2 (posiciones 0, 1, 2), columnas 0 a 3
print("Filas 0–2, columnas 0–3:")
print(df.iloc[0:3, 0:4])

In [ ]:
# .loc — por etiqueta de índice y nombre de columna
print("=== .loc — por etiqueta ===")

# Con índice numérico (defecto), .loc usa el número como etiqueta
# Fila con índice 3, columnas por nombre
print("\nFila índice 3, columnas seleccionadas:")
print(df.loc[3, ['paciente_id', 'hemoglobina_g_dL', 'hematocrito_pct', 'diagnostico']])

In [ ]:
# La diferencia crítica: slicing en .loc es INCLUSIVO en ambos extremos
# En .iloc el extremo derecho es EXCLUSIVO (como Python y Java)

print("Slicing .loc [0:2]  → filas con índice 0, 1 Y 2 (inclusivo):")
print(df.loc[0:2, 'paciente_id'])

print("\nSlicing .iloc [0:2] → posiciones 0 y 1 (exclusivo en extremo derecho):")
print(df.iloc[0:2, 0])   # columna 0 = paciente_id

In [ ]:
# Uso más común de .loc: filtro booleano + selección de columnas
# Esta es la operación que más vas a escribir en análisis real

# Pacientes con creatinina elevada (> 1.5 mg/dL — por encima del límite canino)
# y seleccionar solo las columnas clínicamente relevantes

azotemia = df['creatinina_mg_dL'] > 1.5

resultado_azotemia = df.loc[azotemia, ['paciente_id', 'bun_mg_dL', 
                                        'creatinina_mg_dL', 'diagnostico']]
print("Pacientes con creatinina elevada (posible azotemia):")
print(resultado_azotemia)

In [ ]:
# Combinar múltiples condiciones con .loc
# Pacientes con alteración renal Y hepática simultánea

alteracion_renal    = df['creatinina_mg_dL'] > 1.5
alteracion_hepatica = df['alt_U_L'] > 80      # ALT > 80 U/L en caninos = elevada

perfil_multiorgan = df.loc[
    alteracion_renal & alteracion_hepatica,
    ['paciente_id', 'creatinina_mg_dL', 'bun_mg_dL', 'alt_U_L', 'diagnostico']
]

print("Pacientes con compromiso renal Y hepático:")
if len(perfil_multiorgan) > 0:
    print(perfil_multiorgan)
else:
    print("Ningún paciente cumple ambos criterios simultáneamente")

## 6. Operaciones sobre columnas — transformar datos clínicos

Las columnas de un DataFrame son Series, y las operaciones sobre ellas son vectorizadas.  
Esto significa que puedes calcular nuevas variables clínicas en una sola línea.

In [ ]:
# Crear columnas calculadas — operaciones vectorizadas sobre columnas existentes

# HCM calculado: (Hemoglobina / Eritrocitos) × 10
df['HCM_calculado_pg'] = (df['hemoglobina_g_dL'] / df['eritrocitos_mill_uL']) * 10

# Razón BUN/Creatinina — indicador de función renal pre-renal vs renal
df['ratio_bun_creatinina'] = df['bun_mg_dL'] / df['creatinina_mg_dL']

# Neutrófilos absolutos: leucocitos × (neutrófilos% / 100)
df['neutrofilos_abs_mil_uL'] = df['leucocitos_mil_uL'] * (df['neutrofilos_pct'] / 100)

print("Columnas nuevas agregadas:")
df[['paciente_id', 'HCM_calculado_pg', 
    'ratio_bun_creatinina', 'neutrofilos_abs_mil_uL']].round(2)

In [ ]:
# Columna categórica derivada de condición — usando np.where o pd.cut

# Clasificación de hematocrito en categorías clínicas
df['estado_hematocrito'] = np.where(
    df['hematocrito_pct'] < 25.0, 'Anemia severa',
    np.where(
        df['hematocrito_pct'] < 37.0, 'Anemia',
        np.where(
            df['hematocrito_pct'] <= 55.0, 'Normal',
            'Policitemia'
        )
    )
)

print("Distribución por estado de hematocrito:")
print(df['estado_hematocrito'].value_counts())

In [ ]:
# pd.cut() — categorización por rangos (bins)
# Útil para agrupar valores continuos en categorías clínicas con etiquetas claras

df['categoria_leucocitos'] = pd.cut(
    df['leucocitos_mil_uL'],
    bins   = [0, 6.0, 17.0, 30.0, float('inf')],
    labels = ['Leucopenia', 'Normal', 'Leucocitosis', 'Leucocitosis marcada']
)

print("Distribución de leucocitos por categoría:")
print(df['categoria_leucocitos'].value_counts())

In [ ]:
# Ordenar por columna — muy útil para identificar valores extremos
print("Top 5 pacientes por creatinina (mayor a menor):")
df[['paciente_id', 'creatinina_mg_dL', 'bun_mg_dL', 'diagnostico']]\
    .sort_values('creatinina_mg_dL', ascending=False)\
    .head()

In [ ]:
# Renombrar columnas — útil para limpiar nombres importados de sistemas externos
df_renombrado = df.rename(columns={
    'bun_mg_dL':        'urea_nitrogenada_mg_dL',
    'alt_U_L':          'alanina_aminotransferasa_U_L'
})

print("Columnas renombradas:")
print([c for c in df_renombrado.columns if 'urea' in c or 'alanina' in c])

## 7. groupby — análisis por grupos clínicos

`groupby` es el equivalente a `GROUP BY` en SQL.  
Permite calcular estadísticas por subgrupo: por especie, por diagnóstico, por raza, etc.

In [ ]:
# Estadísticas de hemoglobina y hematocrito por sexo
resumen_por_sexo = df.groupby('sexo')[['hemoglobina_g_dL', 
                                        'hematocrito_pct']].agg(['mean', 'std']).round(2)
print("Hemoglobina y hematocrito por sexo:")
print(resumen_por_sexo)

In [ ]:
# Media de analitos bioquímicos por estado de hematocrito
resumen_bioquimica = df.groupby('estado_hematocrito')[
    ['glucosa_mg_dL', 'bun_mg_dL', 'creatinina_mg_dL', 'alt_U_L']
].mean().round(2)

print("Perfil bioquímico promedio por estado de hematocrito:")
print(resumen_bioquimica)

In [ ]:
# 1. Columna booleana para cada criterio de alerta
df['alerta_anemia']          = df['hematocrito_pct']    < 37.0
df['alerta_azotemia']        = df['creatinina_mg_dL']   > 1.5
df['alerta_leucocitosis']    = df['leucocitos_mil_uL']  > 17.0
df['alerta_trombocitopenia'] = df['plaquetas_mil_uL']   < 200.0

# 2. Contar cuántas alertas tiene cada paciente
# Suma de booleanos por fila (axis=1 = sumar a lo largo de columnas)
columnas_alerta = ['alerta_anemia', 'alerta_azotemia', 
                   'alerta_leucocitosis', 'alerta_trombocitopenia']

df['n_alertas'] = df[columnas_alerta].sum(axis=1)

# 3. Columna tiene_alerta
df['tiene_alerta'] = df['n_alertas'] > 0

# 4. Pacientes con al menos una alerta
cols_reporte = ['paciente_id', 'hematocrito_pct', 'creatinina_mg_dL',
                'leucocitos_mil_uL', 'plaquetas_mil_uL', 
                'n_alertas', 'diagnostico']

print("=== REPORTE DE ALERTAS ===")
reporte = df.loc[df['tiene_alerta'], cols_reporte]\
            .sort_values('n_alertas', ascending=False)
print(reporte.to_string(index=False))

# 5. Distribución de alertas
print("\n=== DISTRIBUCIÓN DE ALERTAS ===")
distribucion = df.groupby('n_alertas')['paciente_id'].count()
distribucion.name = 'n_pacientes'
print(distribucion)

total_alertados = df['tiene_alerta'].sum()
print(f"\nTotal de pacientes con al menos una alerta: {total_alertados} de {len(df)}")